# Testing delle funzioni effettive baselines

## Setup pipeline dati

In [1]:
from data_pipeline import *

#split and clean
df = load_data('data/data-ready-def.xlsx')
train_df, valid_df, test_df = split_data(df)  #i tre set splittati

train_df_clean = clean_data(train_df)
valid_df_clean = clean_data(valid_df)
test_df_clean = clean_data(test_df)

# print(train_df_clean.shape)
# print(valid_df_clean.shape)
# print(test_df_clean.shape)

# print((train_df_clean['Amount of irrigation'] < 0).sum())
# print((valid_df_clean['Amount of irrigation'] < 0).sum())
# print((test_df_clean['Amount of irrigation'] < 0).sum())

#normalize
train_df_norm, valid_df_norm, test_df_norm, train_df_min, train_df_max = normalize(train_df_clean, valid_df_clean, test_df_clean)

columns = train_df_norm.columns.drop(['Date', 'year'])

# print(train_df_norm[columns].min())
# print(train_df_norm[columns].max())
# print('\n\n')
# print(valid_df_norm[columns].min())
# print(valid_df_norm[columns].max())
# print('\n\n')
# print(test_df_norm[columns].min())
# print(test_df_norm[columns].max())

#create_sequence
train_X, train_y = create_sequences(train_df_norm, 20, 1)
valid_X, valid_y = create_sequences(valid_df_norm, 20, 1)
test_X, test_y = create_sequences(test_df_norm, 20, 1)

# print(train_X.shape)
# print(train_y.shape)
# print("\n\n")
# print(valid_X.shape)
# print(valid_y.shape)
# print("\n\n")
# print(test_X.shape)
# print(test_y.shape)

## Baselines

In [2]:
from baselines import *
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

#numero i giorni dei tre set (NOTA: lavoro con i set non normalizzati, poiché Climatology è una semplice media di valori,
#vale lo stesso anche per Persistence e Seasonal Naïve)
train_df_clean['day of season'] = train_df_clean.groupby('year').cumcount() + 1
valid_df_clean['day of season'] = valid_df_clean.groupby('year').cumcount() + 1
test_df_clean['day of season'] = test_df_clean.groupby('year').cumcount() + 1

### Climatology

In [3]:
#prova con valid e test
valid_pred_climatology = climatology_pred(train_df_clean, valid_df_clean)
test_pred_climatology = climatology_pred(train_df_clean, test_df_clean)

valid_mae, valid_rmse, valid_r2 = evaluate(valid_df_clean['Amount of irrigation'], valid_pred_climatology)
test_mae, test_rmse, test_r2 = evaluate(test_df_clean['Amount of irrigation'], test_pred_climatology)

print(valid_mae, valid_rmse, valid_r2)
print(test_mae, test_rmse, test_r2)

1.809398372420864 2.178526078105984 -0.006249411469310129
1.725860348583878 2.085190486898455 0.033064517065183874


### Persistence

In [4]:
valid_pred_persistence = persistence_pred(valid_df_clean, 1)
test_pred_persistence = persistence_pred(test_df_clean, 1)

In [5]:
valid_mask = valid_pred_persistence.notna() #filtro i valori diversi da NaN e li salvo
test_mask = test_pred_persistence.notna()

valid_mae, valid_rmse, valid_r2 = evaluate(valid_df_clean['Amount of irrigation'][valid_mask], valid_pred_persistence[valid_mask])
test_mae, test_rmse, test_r2 = evaluate(test_df_clean['Amount of irrigation'][test_mask], test_pred_persistence[test_mask])

print(valid_mae, valid_rmse, valid_r2)
print(test_mae, test_rmse, test_r2)

1.7032752192982457 2.45591304155855 -0.27075351888623134
1.4111521929824562 2.0490527736197883 0.06991760934631408


### Seasonal Naïve

In [6]:
valid_pred_seasonal = seasonal_pred(valid_df_clean)
test_pred_seasonal = seasonal_pred(test_df_clean)

In [7]:
valid_mask = valid_pred_seasonal.notna() #filtro i valori diversi da NaN e li salvo
test_mask = test_pred_seasonal.notna() #filtro i valori diversi da NaN e li salvo

valid_mae, valid_rmse, valid_r2 = evaluate(valid_df_clean['Amount of irrigation'][valid_mask], valid_pred_seasonal[valid_mask])
test_mae, test_rmse, test_r2 = evaluate(test_df_clean['Amount of irrigation'][test_mask], test_pred_seasonal[test_mask])

print(valid_mae, valid_rmse, valid_r2)
print(test_mae, test_rmse, test_r2)

2.4892781045751633 3.2357503656277524 -1.24794718740762
2.0536571895424838 2.6821279197694015 -0.8222550249932761


### Linear Regression

In [8]:
target_min = train_df_min['Amount of irrigation']
target_max = train_df_max['Amount of irrigation']

lr_results = fit_and_evaluate(LinearRegression(), train_X, train_y, valid_X, valid_y, test_X, test_y, target_min, target_max)
print_metrics("Linear Regression", *lr_results)

--- Linear Regression ---
Valid: MAE=1.73989  RMSE=2.17843  R²=0.0728
Test:  MAE=1.68596  RMSE=2.03110  R²=0.1567



### XGBoost

In [9]:
gb_results = fit_and_evaluate(XGBRegressor(random_state=42), train_X, train_y, valid_X, valid_y, test_X, test_y, target_min, target_max)
print_metrics("Gradient Boosting", *gb_results)

--- Gradient Boosting ---
Valid: MAE=1.69929  RMSE=2.13442  R²=0.1099
Test:  MAE=1.62125  RMSE=2.03599  R²=0.1527



## Baselines unite

In [12]:
# 1. Setup pipeline completa
df = load_data('data/data-ready-def.xlsx')
train_df, valid_df, test_df = split_data(df)

train_df_clean = clean_data(train_df)
valid_df_clean = clean_data(valid_df)
test_df_clean = clean_data(test_df)

train_df_norm, valid_df_norm, test_df_norm, train_df_min, train_df_max = normalize(train_df_clean, valid_df_clean, test_df_clean)

train_X, train_y = create_sequences(train_df_norm, 20, 1)
valid_X, valid_y = create_sequences(valid_df_norm, 20, 1)
test_X, test_y = create_sequences(test_df_norm, 20, 1)

target_col = 'Amount of irrigation'
target_min = train_df_min[target_col]
target_max = train_df_max[target_col]

# 'day of season' serve solo a climatology/persistence/seasonal — lavorano su train_df_clean ecc.,
# che ora restano correttamente in mm anche dopo normalize() (grazie al fix col .copy())
train_df_clean['day of season'] = train_df_clean.groupby('year').cumcount() + 1
valid_df_clean['day of season'] = valid_df_clean.groupby('year').cumcount() + 1
test_df_clean['day of season'] = test_df_clean.groupby('year').cumcount() + 1

# 2. Climatology
climatology_valid_pred = climatology_pred(train_df_clean, valid_df_clean)
climatology_test_pred = climatology_pred(train_df_clean, test_df_clean)

climatology_valid_mae, climatology_valid_rmse, climatology_valid_r2 = evaluate(valid_df_clean[target_col], climatology_valid_pred)
climatology_test_mae, climatology_test_rmse, climatology_test_r2 = evaluate(test_df_clean[target_col], climatology_test_pred)

# 3. Persistence
persistence_valid_pred = persistence_pred(valid_df_clean, H=1)
persistence_test_pred = persistence_pred(test_df_clean, H=1)

valid_mask = persistence_valid_pred.notna()
test_mask = persistence_test_pred.notna()
persistence_valid_mae, persistence_valid_rmse, persistence_valid_r2 = evaluate(valid_df_clean[target_col][valid_mask], persistence_valid_pred[valid_mask])
persistence_test_mae, persistence_test_rmse, persistence_test_r2 = evaluate(test_df_clean[target_col][test_mask], persistence_test_pred[test_mask])

# 4. Seasonal naïve
seasonal_valid_pred = seasonal_pred(valid_df_clean)
seasonal_test_pred = seasonal_pred(test_df_clean)

valid_mask = seasonal_valid_pred.notna()
test_mask = seasonal_test_pred.notna()
seasonal_valid_mae, seasonal_valid_rmse, seasonal_valid_r2 = evaluate(valid_df_clean[target_col][valid_mask], seasonal_valid_pred[valid_mask])
seasonal_test_mae, seasonal_test_rmse, seasonal_test_r2 = evaluate(test_df_clean[target_col][test_mask], seasonal_test_pred[test_mask])

# 5. Linear regression e Gradient Boosting
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

lr_valid_mae, lr_valid_rmse, lr_valid_r2, lr_test_mae, lr_test_rmse, lr_test_r2 = fit_and_evaluate(
    LinearRegression(), train_X, train_y, valid_X, valid_y, test_X, test_y, target_min, target_max
)
gb_valid_mae, gb_valid_rmse, gb_valid_r2, gb_test_mae, gb_test_rmse, gb_test_r2 = fit_and_evaluate(
    XGBRegressor(random_state=42), train_X, train_y, valid_X, valid_y, test_X, test_y, target_min, target_max
)

# 6. Skill score (riferimento: climatology)
def skill_score(mae_model, mae_reference):
    return 1 - (mae_model / mae_reference)

skill_persistence_valid = skill_score(persistence_valid_mae, climatology_valid_mae)
skill_persistence_test = skill_score(persistence_test_mae, climatology_test_mae)
skill_seasonal_valid = skill_score(seasonal_valid_mae, climatology_valid_mae)
skill_seasonal_test = skill_score(seasonal_test_mae, climatology_test_mae)
skill_lr_valid = skill_score(lr_valid_mae, climatology_valid_mae)
skill_lr_test = skill_score(lr_test_mae, climatology_test_mae)
skill_gb_valid = skill_score(gb_valid_mae, climatology_valid_mae)
skill_gb_test = skill_score(gb_test_mae, climatology_test_mae)

# 7. Assemblo la tabella finale
results = pd.DataFrame([
    {'baseline': 'Climatology', 'mae_valid': climatology_valid_mae, 'rmse_valid': climatology_valid_rmse, 'r2_valid': climatology_valid_r2,
     'mae_test': climatology_test_mae, 'rmse_test': climatology_test_rmse, 'r2_test': climatology_test_r2, 'skill_valid': None, 'skill_test': None},
    {'baseline': 'Persistence', 'mae_valid': persistence_valid_mae, 'rmse_valid': persistence_valid_rmse, 'r2_valid': persistence_valid_r2,
     'mae_test': persistence_test_mae, 'rmse_test': persistence_test_rmse, 'r2_test': persistence_test_r2, 'skill_valid': skill_persistence_valid, 'skill_test': skill_persistence_test},
    {'baseline': 'Seasonal naive', 'mae_valid': seasonal_valid_mae, 'rmse_valid': seasonal_valid_rmse, 'r2_valid': seasonal_valid_r2,
     'mae_test': seasonal_test_mae, 'rmse_test': seasonal_test_rmse, 'r2_test': seasonal_test_r2, 'skill_valid': skill_seasonal_valid, 'skill_test': skill_seasonal_test},
    {'baseline': 'Linear regression', 'mae_valid': lr_valid_mae, 'rmse_valid': lr_valid_rmse, 'r2_valid': lr_valid_r2,
     'mae_test': lr_test_mae, 'rmse_test': lr_test_rmse, 'r2_test': lr_test_r2, 'skill_valid': skill_lr_valid, 'skill_test': skill_lr_test},
    {'baseline': 'Gradient boosting', 'mae_valid': gb_valid_mae, 'rmse_valid': gb_valid_rmse, 'r2_valid': gb_valid_r2,
     'mae_test': gb_test_mae, 'rmse_test': gb_test_rmse, 'r2_test': gb_test_r2, 'skill_valid': skill_gb_valid, 'skill_test': skill_gb_test},
])

import os
os.makedirs('results', exist_ok=True)
results.to_csv('results/baselines_H1.csv', index=False)
print(results)

            baseline  mae_valid  rmse_valid  r2_valid  mae_test  rmse_test  \
0        Climatology   1.809398    2.178526 -0.006249  1.725860   2.085190   
1        Persistence   1.703275    2.455913 -0.270754  1.411152   2.049053   
2     Seasonal naive   2.489278    3.235750 -1.247947  2.053657   2.682128   
3  Linear regression   1.739891    2.178430  0.072769  1.685965   2.031097   
4  Gradient boosting   1.699288    2.134420  0.109856  1.621254   2.035993   

    r2_test  skill_valid  skill_test  
0  0.033065          NaN         NaN  
1  0.069918     0.058651    0.182349  
2 -0.822255    -0.375749   -0.189932  
3  0.156725     0.038414    0.023116  
4  0.152655     0.060855    0.060611  
